# 🏋️ Module 2.6: Exercises

Test your understanding of MLFlow Tracking with these exercises.

**Instructions**: Read each exercise, try it yourself, then expand the solution cell to check your work.

---

In [ ]:
# Setup — run this first
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

mlflow.autolog(disable=True)
mlflow.set_experiment("02_exercises")

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Ready!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

---

## Exercise 1: Basic Logging

Train a `LogisticRegression` model with `max_iter=500` and `C=0.5`. Log:
- All parameters (including `random_state=42`)
- Accuracy and F1 score as metrics
- Tags: `author` (your name), `model_family` ("linear")

Name the run `"exercise_1_logreg"`.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — Exercise 1
with mlflow.start_run(run_name="exercise_1_logreg"):
    params = {"max_iter": 500, "C": 0.5, "random_state": 42}
    mlflow.log_params(params)
    
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
    
    mlflow.set_tags({"author": "sujat", "model_family": "linear"})
    
    print(f"✅ Accuracy: {accuracy_score(y_test, y_pred):.4f}")

---

## Exercise 2: Log Artifacts

Train a `RandomForestClassifier` and log:
1. A **confusion matrix plot** as an artifact in the `plots/` subdirectory
2. A **JSON config file** using `mlflow.log_dict()`
3. A **text summary** using `mlflow.log_text()`

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — Exercise 2
import os
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

with mlflow.start_run(run_name="exercise_2_artifacts"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # 1. Confusion matrix plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=wine.target_names, ax=ax, cmap='Blues'
    )
    plt.tight_layout()
    fig.savefig("cm.png", dpi=150)
    mlflow.log_artifact("cm.png", artifact_path="plots")
    plt.show()
    os.remove("cm.png")
    
    # 2. JSON config
    mlflow.log_dict(
        {"model": "RandomForest", "n_estimators": 100, "dataset": "wine"},
        "config/experiment_config.json"
    )
    
    # 3. Text summary
    mlflow.log_text(
        f"Model: RandomForest\nAccuracy: {accuracy_score(y_test, y_pred):.4f}",
        "summary.txt"
    )
    
    print("✅ All artifacts logged!")

---

## Exercise 3: Step-Based Metrics

Simulate a training loop of **30 epochs** that logs `train_loss` and `val_loss` at each step. Use exponential decay with some noise for realism.

After the run, check the MLFlow UI — you should see smooth training curves.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — Exercise 3
np.random.seed(42)

with mlflow.start_run(run_name="exercise_3_training_curves"):
    mlflow.log_params({"epochs": 30, "learning_rate": 0.01, "model": "simulated_nn"})
    
    for epoch in range(30):
        train_loss = 2.0 * np.exp(-0.1 * epoch) + np.random.normal(0, 0.03)
        val_loss = 2.0 * np.exp(-0.08 * epoch) + np.random.normal(0, 0.05)
        
        mlflow.log_metric("train_loss", max(0, train_loss), step=epoch)
        mlflow.log_metric("val_loss", max(0, val_loss), step=epoch)
    
    mlflow.log_metric("final_train_loss", train_loss)
    mlflow.log_metric("final_val_loss", val_loss)
    
    print("✅ Training curves logged! Check the UI.")

---

## Exercise 4: Model Comparison with Autolog

Using `mlflow.autolog()`, train these 4 models and compare results:
1. `DecisionTreeClassifier(max_depth=5)`
2. `RandomForestClassifier(n_estimators=100)`
3. `GradientBoostingClassifier(n_estimators=100)`
4. `LogisticRegression(max_iter=1000)`

After training, use `mlflow.search_runs()` to find the best model by accuracy.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — Exercise 4
mlflow.autolog()

models = [
    ("DecisionTree", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("GradientBoosting", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
]

for name, model in models:
    with mlflow.start_run(run_name=f"ex4_{name}"):
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        mlflow.set_tag("model_name", name)
        print(f"  {name}: {score:.4f}")

# Find best
runs = mlflow.search_runs(
    experiment_names=["02_exercises"],
    filter_string="tags.model_name != ''",
    order_by=["metrics.training_accuracy_score DESC"]
)

if 'tags.model_name' in runs.columns and 'metrics.training_accuracy_score' in runs.columns:
    print(f"\n🏆 Best model: {runs.iloc[0].get('tags.model_name', 'N/A')}")

mlflow.autolog(disable=True)

---

## Exercise 5: Nested HPO Sweep

Create a **parent run** called `"exercise_5_hpo"` that runs a hyperparameter sweep over `RandomForestClassifier` with:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [3, 7, 15]

Each combination should be a **nested child run**. Log the best accuracy on the parent.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION — Exercise 5
import itertools

n_estimators_list = [50, 100, 200]
max_depth_list = [3, 7, 15]

with mlflow.start_run(run_name="exercise_5_hpo"):
    mlflow.set_tag("purpose", "hpo_sweep")
    mlflow.log_param("model_type", "RandomForest")
    
    best_acc = 0
    best_config = {}
    
    for n_est, depth in itertools.product(n_estimators_list, max_depth_list):
        with mlflow.start_run(run_name=f"n{n_est}_d{depth}", nested=True):
            params = {"n_estimators": n_est, "max_depth": depth, "random_state": 42}
            mlflow.log_params(params)
            
            model = RandomForestClassifier(**params)
            model.fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))
            mlflow.log_metric("accuracy", acc)
            
            if acc > best_acc:
                best_acc = acc
                best_config = params
    
    mlflow.log_metric("best_accuracy", best_acc)
    mlflow.log_params({f"best_{k}": v for k, v in best_config.items()})
    
    print(f"🏆 Best: accuracy={best_acc:.4f} with {best_config}")

---

## 🎓 Module 2 Complete!

You've mastered the MLFlow Tracking API:
- ✅ Logging parameters, metrics (including step-based), and tags
- ✅ Logging artifacts (files, plots, directories, text, JSON)
- ✅ Using autolog for automatic tracking
- ✅ Organizing experiments with nested runs

**Next up**: Module 3 — Experiments & the MLFlow UI →